# The field-realistic section: geology in, seismic out

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mosesfalowo/ogbonlab-notebooks/blob/main/notebooks/ssm_field_realistic_section.ipynb)


Generated by [OgbonLab](https://ogbonlab.com), Synthetic Seismic Modeling course.

**Assumptions**: The processed, migrated-image idealization: per-trace 1D convolution in time of a geologically built earth, the same field earth every Ask Ogbon starter shares. Edit the parameters cell and run again.

**Workflow**:
1. Fold thin sand and shale beds over a regional dip; cut them with normal faults, the largest a listric growth fault with a rollover
2. Uplift and erode the crest at an angular unconformity, so the reservoir sand subcrops, wedges out and holds gas over a flat contact
3. Drape the overburden, with a channel; give every cell the compaction velocity of its deepest burial, and Gardner density
4. Convert depth to time trace by trace, and image with a Ricker whose peak falls with time as constant Q says
5. Add noise in the band of the data

In [ ]:
# The field-realistic section: geology in, seismic out
# Generated by OgbonLab (ogbonlab.com), Synthetic Seismic Modeling course.
"""The field earth every Ask Ogbon starter shares, built in the open and then imaged.

The earth is built in geological order: thin sand and shale beds with a regional
dip, folded into an asymmetric anticline; normal faults, the largest a listric
growth fault whose hanging wall rolls over into it; uplift and an angular
unconformity that cuts the crest; a reservoir sand that subcrops beneath the
unconformity, wedges out against it, shales out to the left and holds gas over a
flat contact; a draped overburden with a channel. Velocities follow compaction at
each rock's deepest burial; density is Gardner's; the reservoir and its seal are
the book's standard rocks (Section 3.3).
Then the section: each trace from depth to two-way time by vertical rays, the
reflectivity, a Ricker whose peak falls with time as constant Q says, and noise
in the band of the data. This is the processed, migrated-image idealization:
1D convolution trace by trace (Sections 2.2 and 3.4), no diffractions and no
multiples (the finite-difference notebook has the first, the multiples starter
the second). Every number it prints is computed here.
"""
import numpy as np
import matplotlib.pyplot as plt

# --- parameters ---
SEED = 7          # change it: another draw of the beds' thicknesses and rocks, the same structure
F_HZ = 30.0       # Ricker peak at the surface (Hz)
Q = 80.0          # the earth's quality factor: the peak falls with time
SNR = 6.0         # signal-to-noise of the final image
DT_MS = 2.0       # sample interval (ms)
T_MAX_MS = 1100.0 # record length (ms)

In [ ]:
def report(name, value, fmt='%.6f'):
    """Print one result as name = value, the line Figure 12.5 checks against the book's kernel."""
    print(('%s = ' + fmt) % (name, value))

def style(ax, title, xlabel, ylabel, grid=True):
    """The starters' one look: a title, both axes labelled with their units, a light grid."""
    ax.set_title(title, fontsize=10)
    ax.set_xlabel(xlabel)
    ax.set_ylabel(ylabel)
    if grid:
        ax.grid(True, alpha=0.3)

# the book's standard rocks (SSM.STANDARD, Section 3.3): P velocity (m/s), S velocity (m/s), density (g/cc)
ROCKS = {
    'shale': (2800, 1241.3793103448277, 2.2521031547808517),
    'brine sand': (2889.973997793982, 1504.4719169684909, 2.2545487499999997),
    'oil sand': (2543.1993596875604, 1529.5017037311773, 2.1813626422963988),
    'gas sand': (2341.92404704112, 1582.4208998550723, 2.0379042775100316)
}

# --- the Ogbon field earth: one seeded 2D earth that every starter shares ---
# In geological order: thin sand and shale beds with a regional dip, folded into an
# asymmetric anticline and cut by normal faults, the largest a listric growth fault
# (its throw grows with depth and its hanging wall rolls over into it); uplift, and an
# angular unconformity that truncates the crest; the reservoir sand subcrops beneath
# it, wedging out against it, shaling out to the left, sealed by a shale cap and
# holding gas above a flat contact; then a draped overburden with a channel. Each
# cell's velocity follows the
# compaction trend at its deepest burial (the rocks under the unconformity were once
# buried deeper), sands soft when shallow and hard when deep; Gardner's density; the
# mudrock and sandstone lines for S. Units: m, m/s, g/cc; times in ms.
FAULTS = [(0.66, 1.0, 1.4, 380.0), (0.80, -1.0, 0.6, 470.0), (0.22, -1.0, 0.5, 560.0)]   # (x / length, side, size, tip m)
COMPACTION = (1900.0, 1.4)                             # P velocity = 1900 m/s + 1.4 per metre of burial, before lithology
K_RES = 9                                              # the reservoir: bed 9 of the deep package, 70 m thick
SEAL_M = 30.0                                          # the shale cap on the reservoir's top: 30 m, and never under two cells

def field_earth(nx=240, nz=320, dx=15.0, dz=5.0, seed=7, crest=0.45, fold=1.0, dip=1.0, faults=None, throw=1.0,
                unconformity=True, seal=None, upper=None, lower=None, column_m=130.0):
    """The field earth on nz by nx cells of dz by dx (m). seal, upper and lower are (vp, vs, density) of the shale on the
    unconformity and of the reservoir above and below its flat contact, or None for the compaction trend's own rocks."""
    rng = np.random.default_rng(seed)
    x, z = np.arange(nx) * dx, np.arange(nz) * dz
    xn, zz, sc = x / x[-1], z[:, None] + 0.0 * x, nz * dz / 1600.0     # sc: the geometry scales with the grid's depth
    hump = np.exp(-0.5 * ((xn - crest) / np.where(xn < crest, 0.13, 0.25)) ** 2)   # the anticline, steep to the left
    swell = 1.0 + 0.15 * np.sin(2 * np.pi * (0.8 * xn + 0.37 * np.arange(26)[:, None]))   # each bed thickens and thins
    thick = sc * (24.0 + 22.0 * rng.random(26))[:, None] * swell
    thick[K_RES] = sc * 70.0
    beds = sc * 470.0 + np.cumsum(thick, axis=0) - thick + sc * (170.0 * dip * xn - 300.0 * fold * hump)
    if faults is not None:
        fs = [(rng.uniform(0.12, 0.88), rng.choice([-1.0, 1.0]), rng.uniform(0.4, 1.4), rng.uniform(300.0, 600.0))
              for k in range(faults)]
    fault = np.zeros((nz, nx), bool)
    for x0, side, size, tip in (FAULTS if faults is None else fs):
        tip, a = tip * sc, 0.2 / (1200.0 * sc) ** 1.3       # the plane: x / length = x0 + side a (depth - tip) to the 1.3, listric
        plane = np.where(side * (xn - x0) > 0, tip + (np.abs(xn - x0) / a) ** (1 / 1.3), -np.inf)   # its depth at each x
        for k in range(26):
            g = throw * size * sc * 90.0 * np.clip((beds[k] - tip) / (700.0 * sc), 0.0, 1.6)      # growth: deeper beds slip more
            xk = x0 + side * a * np.maximum(beds[k] - tip, 0.0) ** 1.3                          # where the plane cuts the bed
            g = g * (1.0 + 0.6 * np.exp(-np.abs(xn - xk) / 0.04))                                # the rollover into the fault
            beds[k] = beds[k] + np.clip(plane - beds[k], 0.0, g)       # the hanging wall drops; in the heave the bed lies on the plane
        ix = np.rint((x0 + side * a * np.maximum(z - tip, 0.0) ** 1.3) * (nx - 1)).astype(int)
        on = (z > tip) & (ix >= 0) & (ix < nx)
        fault[np.nonzero(on)[0], ix[on]] = True
    beds = np.maximum.accumulate(beds, axis=0)
    if unconformity:                                   # cut just under the reservoir's crest, so it subcrops on both flanks
        unc = beds[K_RES + 1, np.argmax(hump)] + sc * (15.0 + 160.0 * dip * (xn - crest)
                                                       + 10.0 * np.sin(2 * np.pi * (1.3 * xn + 0.2)))
    else:
        unc = beds[0] - sc * 30.0                      # no erosion: the overburden lies on the beds
    fault &= zz > unc                                  # the faults are older than the unconformity
    fr = np.cumsum(0.6 + rng.random(13))
    fr = (fr - fr[0]) / (fr[-1] - fr[0])
    over = [sc * 40.0 + (unc.min() - sc * 90.0) * f + 0.85 * f * f * (unc - unc.min()) - sc * 25.0 * f * hump for f in fr]
    tops = np.maximum.accumulate(np.vstack(over + list(np.maximum(beds, unc))), axis=0)
    lay = (zz[:, None, :] >= tops[None, :, :]).sum(axis=1)                    # layer 13 is the seal; bed k is layer 14 + k
    sand = np.where(lay < 13, rng.random(41)[lay] < 0.45, (lay - 14) % 2 == 1) & (lay > 0) & (lay != 13)
    chan = (zz >= over[5]) & (zz < over[5] + sc * 45.0 * np.exp(-0.5 * ((xn - 0.7) / 0.04) ** 2))
    res = lay == 14 + K_RES
    top = np.where(res, zz, np.inf).min(axis=0)
    cap = (zz < top) & (zz >= top - max(SEAL_M * sc, 2.0 * dz))   # the seal: shale on the sand's top, under a bed or the unconformity
    sand = np.where(cap, False, sand)
    burial = zz + np.where(lay >= 14, sc * 150.0 * hump, 0.0)                # uplift: the deep rocks were buried deeper
    vary = 1.0 + 0.035 * np.clip(rng.standard_normal(41), -1.2, 1.2)        # bed to bed variety
    vp = (COMPACTION[0] + COMPACTION[1] * burial) * vary[lay]
    vp = np.where(sand | chan, vp * (0.86 + 0.1 * np.minimum(burial / 1000.0, 1.8)), vp)
    vp = np.where(chan, vp * 0.92, vp)                 # the channel cuts a horizon and is filled with a soft sand
    sand |= chan
    vs = np.where(sand, 0.804 * vp - 856.0, (vp - 1360.0) / 1.16)            # the sandstone and mudrock lines (Castagna)
    den = 1.741 * (vp / 1000.0) ** 0.25                                      # Gardner
    ntg = np.clip((xn - 0.12) / 0.12, 0.0, 1.0)                              # the reservoir shales out to the left
    contact = float(top.min()) + column_m * sc         # the flat contact, column_m under the shallowest sand
    for rock, on in ((seal, (lay == 13) | cap), (upper, res & (zz < contact)), (lower, res & (zz >= contact))):
        if rock is not None:
            w = 1.0 if rock is seal else ntg
            vp, vs, den = [np.where(on, w * rock[i] + (1 - w) * q, q) for i, q in enumerate((vp, vs, den))]
    return dict(x=x, z=z, vp=vp, vs=vs, den=den, lay=lay, sand=sand, res=res, fault=fault, unc=unc, top=top,
                contact=contact, ntg=ntg, dx=dx, dz=dz, chan=chan, cap=cap, gas=res & (zz < contact) & (ntg > 0.5))

def field_time(m, dt_ms, t_max_ms):
    """Each trace from depth to two-way time by vertical rays: every cell array of m on a dt_ms grid to t_max_ms."""
    t_top = 2000.0 * np.cumsum(m['dz'] / m['vp'], axis=0) - 2000.0 * m['dz'] / m['vp']
    t = np.arange(0.0, t_max_ms, dt_ms)
    pick = np.array([np.searchsorted(t_top[:, i], t, 'right') - 1 for i in range(t_top.shape[1])]).T
    cols = np.arange(t_top.shape[1])[None, :]
    out = dict((k, m[k][pick, cols]) for k in ('vp', 'vs', 'den', 'lay', 'sand', 'res', 'gas'))
    seen = np.vstack([0 * cols, np.cumsum(m['fault'], axis=0)])                 # fault cells above each row
    after = np.maximum(np.vstack([pick[1:], pick[-1:] + 1]), pick + 1)
    out['fault'] = seen[np.minimum(after, seen.shape[0] - 1), cols] > seen[pick, cols]   # a fault cell inside the sample
    out['t'], out['t_top'] = t, t_top
    return out

def reflectivity(ai):
    """Normal-incidence coefficient at the top of every sample, down axis 0 (Section 2.1)."""
    rc = np.zeros_like(ai)
    rc[1:] = (ai[1:] - ai[:-1]) / (ai[1:] + ai[:-1])
    return rc

def q_peak_hz(f0, t_s, q):
    """Peak (Hz) of a Ricker of peak f0 after t_s seconds through constant Q: the root of 2 f f / (f0 f0) + (pi t / Q) f = 2."""
    b = np.pi * t_s / q
    return (-b + np.sqrt(b * b + 16.0 / f0 ** 2)) * f0 ** 2 / 4.0

def field_image(rc, dt_ms, f0, q=80.0, snr=None, seed=11):
    """Every trace convolved with a zero-phase Ricker whose peak falls with time as constant q says, plus coloured noise."""
    nt, dt = rc.shape[0], dt_ms * 1e-3
    f = np.fft.rfftfreq(2 * nt, dt)[:, None]
    def band(fp):                                      # the Ricker's spectrum (Section 1.1), over dt for a sum of samples
        return 2.0 / np.sqrt(np.pi) * f ** 2 / fp ** 3 * np.exp(-(f / fp) ** 2) / dt
    def filt(a, fp):
        return np.fft.irfft(np.fft.rfft(a, 2 * nt, axis=0) * band(fp), 2 * nt, axis=0)[:nt]
    fp = q_peak_hz(f0, np.arange(nt) * dt, q)
    knots = np.linspace(fp.min(), fp.max(), 4 if fp.max() - fp.min() > 1e-9 else 1)
    step = max((fp.max() - fp.min()) / 3.0, 1e-9)
    out = sum(np.clip(1.0 - np.abs(fp - k) / step, 0.0, 1.0)[:, None] * filt(rc, k) for k in knots)
    if snr:
        noise = filt(np.random.default_rng(seed).standard_normal(rc.shape), fp.mean())
        noise = (noise + np.roll(noise, 1, 1) + np.roll(noise, -1, 1)) / 3.0          # coherent over a few traces
        out = out + noise * np.sqrt((out ** 2).mean() / (noise ** 2).mean()) / snr
    return out

def fieldnote(name, value, fmt='%.3f'):
    """Print a number of the field earth as [field] name = value: computed here, not by Figure 12.5."""
    print(('[field] %s = ' + fmt) % (name, value))

def show(ax, a, x_m, y, title, ylabel='two-way time (ms)', clip=None):
    """A section as an image, black for positive, clipped at clip (or at its 99th percentile): km across, time or depth down."""
    c = clip if clip else np.percentile(np.abs(a), 99.0) + 1e-12
    ax.imshow(a, cmap='gray_r', vmin=-c, vmax=c, aspect='auto', extent=[x_m[0] / 1e3, x_m[-1] / 1e3, y[-1], y[0]])
    style(ax, title, 'distance (km)', ylabel, grid=False)


def outline(ax, e, mask, **kw):
    """The edge of a mask of cells, drawn over a depth panel in km and m."""
    ax.contour(e['x'] / 1e3, e['z'], mask.astype(float), [0.5], **kw)

def depth_panel(ax, e, title, deep_only=False):
    """P velocity in depth on one colour scale for every panel; deep_only leaves out what lies above the old beds."""
    v = np.ma.masked_where(e['lay'] < 14, e['vp']) if deep_only else e['vp']
    im = ax.imshow(v / 1e3, cmap='viridis', vmin=1.6, vmax=4.6, aspect='auto',
                   extent=[0, e['x'][-1] / 1e3, e['z'][-1], 0])
    style(ax, title, 'distance (km)', 'depth (m)', grid=False)
    return im


rocks = dict(seal=ROCKS['shale'], upper=ROCKS['gas sand'], lower=ROCKS['brine sand'])
folded = field_earth(seed=SEED, faults=0, unconformity=False)
faulted = field_earth(seed=SEED, unconformity=False)
e = field_earth(seed=SEED, **rocks)
t = field_time(e, DT_MS, T_MAX_MS)
rc = reflectivity(t['vp'] * t['den'])
section = field_image(rc, DT_MS, F_HZ, q=Q, snr=SNR)

# the numbers a reviewer would ask for, every one computed from the earth above
nz_rc = np.abs(rc[np.abs(rc) > 1e-6])
gas_cols = np.nonzero(e['gas'].any(axis=0))[0]
ic = gas_cols[np.argmax(e['gas'][:, gas_cols].sum(axis=0))]
fieldnote('vp_min_mps', e['vp'].min(), '%.0f')
fieldnote('vp_max_mps', e['vp'].max(), '%.0f')
fieldnote('rc_median_abs', np.median(nz_rc))
fieldnote('rc_p99_abs', np.percentile(nz_rc, 99))
fieldnote('fault_cells', e['fault'].sum(), '%d')
fieldnote('reservoir_crest_m', e['top'].min(), '%.0f')
fieldnote('contact_m', e['contact'], '%.0f')
fieldnote('gas_column_thickest_m', e['gas'][:, ic].sum() * e['dz'], '%.0f')
fieldnote('flat_spot_rc', (ROCKS['brine sand'][0] * ROCKS['brine sand'][2] - ROCKS['gas sand'][0] * ROCKS['gas sand'][2])
          / (ROCKS['brine sand'][0] * ROCKS['brine sand'][2] + ROCKS['gas sand'][0] * ROCKS['gas sand'][2]))
fieldnote('peak_hz_at_top', q_peak_hz(F_HZ, 0.0, Q), '%.1f')
fieldnote('peak_hz_at_%.0f_ms' % T_MAX_MS, q_peak_hz(F_HZ, T_MAX_MS * 1e-3, Q), '%.1f')

# --- figure 1: the earth, built in geological order -------------------------------
fig, ax = plt.subplots(2, 2, figsize=(13, 8.2))
depth_panel(ax[0, 0], folded, '1. Beds, a regional dip, an asymmetric anticline', deep_only=True)
depth_panel(ax[0, 1], faulted, '2. Normal faults; the largest grows with depth and rolls over', deep_only=True)
fz, fx = np.nonzero(faulted['fault'])
ax[0, 1].plot(faulted['x'][fx] / 1e3, faulted['z'][fz], 'r.', ms=1)
im = depth_panel(ax[1, 0], e, '3. Uplift, erosion, the overburden; the reservoir holds gas')
ax[1, 0].plot(e['x'] / 1e3, e['unc'], color='gold', lw=1.4)
outline(ax[1, 0], e, e['res'] & (e['ntg'][None, :] > 0.5), colors='w', linewidths=0.8)
outline(ax[1, 0], e, e['gas'], colors='magenta', linewidths=1.6)
outline(ax[1, 0], e, e['chan'], colors='orange', linewidths=1.0)
box = dict(facecolor='white', alpha=0.85, edgecolor='none', pad=1.5)
xg, km = e['x'][ic] / 1e3, e['x'][-1] / 1e3
x0, side, size, tip = FAULTS[0]                     # the growth fault's plane, at 1100 m
xf = (x0 + side * 0.2 * ((1100.0 - tip) / 1200.0) ** 1.3) * km
iw = int(np.argmax(e['res'].any(axis=0) & (e['ntg'] > 0.5)))       # where the sand begins, from the left
chan_z = e['z'][np.nonzero(e['chan'].any(axis=1))[0]].mean()
notes = [('angular unconformity', (0.35, e['unc'][int(0.1 * e['x'].size)]), (0.15, 420)),
         ('gas over a flat contact', (xg, e['contact'] - 10), (xg - 0.7, 1080)),
         ('the reservoir wedges out', (xg + 0.25, e['top'][np.argmin(np.abs(e['x'] / 1e3 - xg - 0.25))]), (1.9, 400)),
         ('the sand shales out', (e['x'][iw] / 1e3, e['top'][iw]), (0.1, 1250)),
         ('channel', (0.7 * km, chan_z), (2.9, 150)),
         ('growth fault', (xf, 1100), (3.0, 1450))]
for text, xy, xyt in notes:
    ax[1, 0].annotate(text, xy, xyt, fontsize=8, bbox=box, arrowprops=dict(arrowstyle='-', lw=0.8))
fig.colorbar(im, ax=ax[1, 0], label='P velocity (km/s)')
for k, (ix, name) in enumerate(((ic, 'through the gas'), (int(0.9 * e['x'].size), 'far down the right flank'))):
    ax[1, 1].plot(e['vp'][:, ix], e['z'], lw=1, label='%s, %.2f km' % (name, e['x'][ix] / 1e3))
ax[1, 1].plot(COMPACTION[0] + COMPACTION[1] * e['z'], e['z'], 'k--', lw=0.8, label='the compaction trend, burial = depth')
ax[1, 1].invert_yaxis()
style(ax[1, 1], '4. Velocity with depth: compaction, uplift, soft gas', 'P velocity (m/s)', 'depth (m)')
ax[1, 1].legend(fontsize=7, loc='lower left')
fig.tight_layout()
plt.show()

# --- figure 2: geology in, seismic out --------------------------------------------
fig, ax = plt.subplots(1, 2, figsize=(13, 4.8))
im = ax[0].imshow(e['vp'] * e['den'] / 1e3, cmap='viridis', aspect='auto', extent=[0, e['x'][-1] / 1e3, e['z'][-1], 0])
fig.colorbar(im, ax=ax[0], label='impedance (km/s x g/cc)')
style(ax[0], 'The earth: impedance in depth', 'distance (km)', 'depth (m)', grid=False)
show(ax[1], section, e['x'], t['t'], 'The section: Q = %.0f, S/N %.0f, 1D convolution trace by trace' % (Q, SNR))
tg = t['t'][t['gas'][:, ic]]
ax[1].annotate('flat spot', (xg, tg.max()), (xg - 0.8, tg.max() + 150), fontsize=8, color='r', bbox=box,
               arrowprops=dict(arrowstyle='-', color='r'))
ax[1].annotate('bright spot', (xg, tg.min()), (xg - 0.9, tg.min() - 120), fontsize=8, color='r', bbox=box,
               arrowprops=dict(arrowstyle='-', color='r'))
fig.tight_layout()
plt.show()

## Where to go next

- The course: https://ogbonlab.com/textbooks/synthetic-seismic-modeling
- More notebooks: https://github.com/mosesfalowo/ogbonlab-notebooks
- Scale up: Devito, SPECFEM3D, Madagascar (course Part 12.3)